# RetailPulse — Retail Pricing & Demand Analytics

## Project Goal

- Analyze retail pricing, demand, competitor positioning, and business performance.
- Understand how price movements are associated with changes in product demand.
- Identify potential pricing opportunities using data-driven analysis.

## Project Objectives

- Clean and validate product-level monthly retail data.
- Analyze price changes and pricing patterns over time.
- Compare demand behavior during price increases, decreases, and unchanged-price periods.
- Evaluate competitor/reference pricing and competitive positioning.
- Analyze category, product, revenue, customer, and time-based performance.
- Apply a simple statistical test to validate an important pricing observation.
- Identify potential products or categories for pricing review.
- Design a simple proposed price experiment to evaluate a future pricing decision.
- Use SQL for business-focused analysis and Power BI for interactive reporting.
- Present actionable pricing insights, recommendations, and analysis limitations.

## 1. Data Understanding

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Load the Dataset

The retail pricing dataset is loaded into a Pandas DataFrame for analysis.

In [2]:
df = pd.read_csv("../data/retail_price.csv")

### Preview the Dataset

The first five rows are displayed to get an initial understanding of the available columns and observations.

In [3]:
df.head()

,product_id,product_category_name,month_year,qty,total_price,freight_price,unit_price,product_name_lenght,product_description_lenght,product_photos_qty,...,comp_1,ps1,fp1,comp_2,ps2,fp2,comp_3,ps3,fp3,lag_price
0,bed1,bed_bath_table,01-05-2017,1,45.95,15.100000,45.95,39,161,2,...,89.9,3.9,15.011897,215.000000,4.4,8.760000,45.95,4.0,15.100000,45.90
1,bed1,bed_bath_table,01-06-2017,3,137.85,12.933333,45.95,39,161,2,...,89.9,3.9,14.769216,209.000000,4.4,21.322000,45.95,4.0,12.933333,45.95
2,bed1,bed_bath_table,01-07-2017,6,275.70,14.840000,45.95,39,161,2,...,89.9,3.9,13.993833,205.000000,4.4,22.195932,45.95,4.0,14.840000,45.95
3,bed1,bed_bath_table,01-08-2017,4,183.80,14.287500,45.95,39,161,2,...,89.9,3.9,14.656757,199.509804,4.4,19.412885,45.95,4.0,14.287500,45.95
4,bed1,bed_bath_table,01-09-2017,2,91.90,15.100000,45.95,39,161,2,...,89.9,3.9,18.776522,163.398710,4.4,24.324687,45.95,4.0,15.100000,45.95


### Dataset Dimensions

The number of rows and columns is checked to understand the overall size of the dataset before performing further analysis.

In [4]:
df.shape

(676, 30)

### Available Variables

The column names are reviewed to identify variables relevant to product characteristics, pricing, demand, competitors, and time.

In [5]:
df.columns

Index(['product_id', 'product_category_name', 'month_year', 'qty',
       'total_price', 'freight_price', 'unit_price', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_score', 'customers', 'weekday', 'weekend', 'holiday', 'month',
       'year', 's', 'volume', 'comp_1', 'ps1', 'fp1', 'comp_2', 'ps2', 'fp2',
       'comp_3', 'ps3', 'fp3', 'lag_price'],
      dtype='object')

### Data Types and Completeness

The dataset structure is inspected to identify the data type of each variable and assess whether the available fields are suitable for pricing, demand, competitor, and time-based analysis.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 676 entries, 0 to 675
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  676 non-null    object 
 1   product_category_name       676 non-null    object 
 2   month_year                  676 non-null    object 
 3   qty                         676 non-null    int64  
 4   total_price                 676 non-null    float64
 5   freight_price               676 non-null    float64
 6   unit_price                  676 non-null    float64
 7   product_name_lenght         676 non-null    int64  
 8   product_description_lenght  676 non-null    int64  
 9   product_photos_qty          676 non-null    int64  
 10  product_weight_g            676 non-null    int64  
 11  product_score               676 non-null    float64
 12  customers                   676 non-null    int64  
 13  weekday                     676 non

## 2. Data Cleaning & Validation

- Check missing values and duplicate records.
- Validate product-month uniqueness and date format.
- Review key pricing and demand variables for invalid or unusual values.
- Investigate outliers before deciding whether they should be removed.

In [7]:
df.isnull().sum()

product_id                    0
product_category_name         0
month_year                    0
qty                           0
total_price                   0
freight_price                 0
unit_price                    0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_score                 0
customers                     0
weekday                       0
weekend                       0
holiday                       0
month                         0
year                          0
s                             0
volume                        0
comp_1                        0
ps1                           0
fp1                           0
comp_2                        0
ps2                           0
fp2                           0
comp_3                        0
ps3                           0
fp3                           0
lag_price                     0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

### Product-Month Granularity Validation

Since the dataset is expected to contain one observation per product per month, the combination of `product_id` and `month_year` is checked for duplicates to verify that each product-month observation is unique.

In [9]:
df.duplicated(subset=["product_id", "month_year"]).sum()

np.int64(0)

### Date Format Standardization

The `month_year` variable was initially stored as an object. It is converted to a datetime format to support chronological sorting and time-based pricing analysis.

In [10]:
df["month_year"] = pd.to_datetime(df["month_year"], format="%d-%m-%Y")
df["month_year"].head()

0   2017-05-01
1   2017-06-01
2   2017-07-01
3   2017-08-01
4   2017-09-01
Name: month_year, dtype: datetime64[ns]

### Business-Rule Validation

- Review the distribution and range of price, quantity, and revenue variables.
- Check for impossible or suspicious business values before analysis.

In [11]:
df[["unit_price", "qty", "total_price"]].describe()

,unit_price,qty,total_price
count,676.000000,676.000000,676.000000
mean,106.496800,14.495562,1422.708728
std,76.182972,15.443421,1700.123100
min,19.900000,1.000000,19.900000
25%,53.900000,4.000000,333.700000
50%,89.900000,10.000000,807.890000
75%,129.990000,18.000000,1887.322500
max,364.000000,122.000000,12095.000000


**Finding**
- No negative or zero values were observed for price, quantity, or revenue.
- Large values will be investigated as potential outliers rather than removed automatically.

### Revenue Field Validation

- Compare the supplied `total_price` with `qty × unit_price`.
- Validate whether revenue can be directly reconstructed from the observed monthly average unit price.

In [12]:
df["calculated_total_price"] = df["qty"] * df["unit_price"]

revenue_match = np.isclose(
    df["total_price"],
    df["calculated_total_price"]
)

pd.Series(revenue_match).value_counts()

True     598
False     78
Name: count, dtype: int64

In [13]:
df.loc[
    ~revenue_match,
    ["product_id", "month_year", "qty", "unit_price",
     "total_price", "calculated_total_price"]
].head()

,product_id,month_year,qty,unit_price,total_price,calculated_total_price
18,garden5,2017-05-01,20,97.588235,1956.0,1951.764706
21,garden5,2017-08-01,7,97.333333,683.0,681.333333
48,garden7,2017-11-01,20,51.322222,1043.6,1026.444444
49,garden7,2017-12-01,10,56.977778,571.8,569.777778
54,garden7,2018-05-01,9,52.900000,477.1,476.100000


**Finding**
- 598 of 676 observations closely matched `qty × unit_price`, while 78 showed differences.
- `total_price` therefore cannot be reconstructed exactly from the observed monthly average `unit_price`.
- The supplied `total_price` is retained as the revenue measure for subsequent analysis.

### Outlier Detection

- Use the IQR method to identify unusually high or low unit prices.
- Investigate flagged observations before deciding whether they should be removed.

In [14]:
Q1 = df["unit_price"].quantile(0.25)
Q3 = df["unit_price"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

price_outliers = df[
    (df["unit_price"] < lower_bound) |
    (df["unit_price"] > upper_bound)
]

price_outliers[["product_id", "product_category_name", "month_year", "unit_price"]].head()

,product_id,product_category_name,month_year,unit_price
339,health5,health_beauty,2017-01-01,349.9
340,health5,health_beauty,2017-02-01,349.9
341,health5,health_beauty,2017-03-01,349.9
342,health5,health_beauty,2017-04-01,349.9
343,health5,health_beauty,2017-05-01,349.9


In [15]:
price_outliers.shape[0]

45

In [16]:
price_outliers[
    [
        "product_id",
        "product_category_name",
        "month_year",
        "unit_price",
        "qty"
    ]
].sort_values("unit_price", ascending=False)

,product_id,product_category_name,month_year,unit_price,qty
358,health5,health_beauty,2018-08-01,364.000000,9
357,health5,health_beauty,2018-07-01,364.000000,11
356,health5,health_beauty,2018-06-01,364.000000,11
355,health5,health_beauty,2018-05-01,352.250000,10
350,health5,health_beauty,2017-12-01,349.900000,7
340,health5,health_beauty,2017-02-01,349.900000,4
354,health5,health_beauty,2018-04-01,349.900000,7
353,health5,health_beauty,2018-03-01,349.900000,17
352,health5,health_beauty,2018-02-01,349.900000,15
351,health5,health_beauty,2018-01-01,349.900000,12


### Quantity Outlier Investigation

- Review unusually high sales quantities to determine whether they represent errors or genuine demand spikes.
- Inspect the product's history before deciding whether to remove the observation.

In [17]:
df.nlargest(
    10,
    "qty"
)[
    [
        "product_id",
        "product_category_name",
        "month_year",
        "qty",
        "unit_price"
    ]
]

,product_id,product_category_name,month_year,qty,unit_price
499,furniture2,furniture_decor,2018-01-01,122,69.995327
277,watches6,watches_gifts,2018-05-01,114,103.792523
503,furniture2,furniture_decor,2018-05-01,92,69.900000
195,garden6,garden_tools,2017-11-01,91,52.406944
164,garden10,garden_tools,2017-11-01,87,53.648281
276,watches6,watches_gifts,2018-04-01,85,105.648625
502,furniture2,furniture_decor,2018-04-01,83,69.900000
257,garden9,garden_tools,2017-11-01,82,53.379104
282,cool3,cool_stuff,2018-01-01,76,29.500000
654,computers2,computers_accessories,2018-02-01,71,78.712281


In [18]:
df[
    df["product_id"] == "furniture2"
][
    ["month_year", "qty", "unit_price", "lag_price", "holiday"]
].sort_values("month_year")

,month_year,qty,unit_price,lag_price,holiday
494,2017-07-01,5,75.000000,74.950000,1
495,2017-08-01,13,75.000000,75.000000,1
496,2017-10-01,5,75.000000,74.950000,2
497,2017-11-01,44,75.000000,75.000000,4
498,2017-12-01,47,75.000000,75.000000,1
499,2018-01-01,122,69.995327,75.000000,2
500,2018-02-01,44,69.900000,69.995327,2
501,2018-03-01,65,69.900000,69.900000,1
502,2018-04-01,83,69.900000,69.900000,1
503,2018-05-01,92,69.900000,69.900000,3


### Outlier Assessment

- IQR screening identified 45 unusually priced observations.
- High-quantity observations were also reviewed at the product level.
- The flagged observations appeared plausible rather than obvious data errors.
- Price and quantity outliers were therefore retained for subsequent analysis.

## 3. Lag Price Validation

- Sort product records chronologically before calculating previous observed prices.
- Compare the supplied `lag_price` with an independently calculated previous price.
- Investigate mismatches before using lag-based price changes.

In [19]:
df = df.sort_values(["product_id", "month_year"])

In [20]:
df = df.sort_values(["product_id", "month_year"]).copy()

df["calculated_lag_price"] = (
    df.groupby("product_id")["unit_price"].shift(1)
)

In [21]:
lag_check = df[df["calculated_lag_price"].notna()].copy()

lag_check["lag_match"] = np.isclose(
    lag_check["lag_price"],
    lag_check["calculated_lag_price"]
)

lag_check["lag_match"].value_counts()

lag_match
True     595
False     29
Name: count, dtype: int64

In [22]:
df["previous_date"] = df.groupby("product_id")["month_year"].shift(1)

df["month_gap"] = (
    (df["month_year"].dt.year - df["previous_date"].dt.year) * 12
    + (df["month_year"].dt.month - df["previous_date"].dt.month)
)

df["month_gap"].value_counts().sort_index()

month_gap
1.0    595
2.0     22
3.0      5
4.0      1
5.0      1
Name: count, dtype: int64

In [23]:
lag_check = df[
    df["calculated_lag_price"].notna()
].copy()

lag_check["lag_match"] = np.isclose(
    lag_check["lag_price"],
    lag_check["calculated_lag_price"]
)

lag_mismatches = lag_check[~lag_check["lag_match"]]

lag_mismatches["month_gap"].value_counts().sort_index()

month_gap
2.0    22
3.0     5
4.0     1
5.0     1
Name: count, dtype: int64

### Lag Price Assessment

- 624 observations had a previous product record available for comparison.
- 595 supplied `lag_price` values matched the independently calculated previous observed price.
- The remaining 29 mismatches occurred only when product histories contained gaps greater than one month.
- `calculated_lag_price` is used for subsequent price-change analysis because its logic is transparent and reproducible.
- It represents the previous available observation, which may not always be the immediately preceding calendar month.

## 4. Price Change Analysis

- Calculate absolute and percentage price changes using the previous available product price.
- Classify observations as price increases, decreases, or no change.
- Examine how frequently different pricing movements occurred.
Two features are calculated:

- `price_change`: Absolute change in the monthly average unit price.
- `price_change_pct`: Percentage change relative to the previous available monthly average unit price.

These metrics will be used to identify price increases, decreases, stable pricing periods, and unusually large price movements.

In [24]:
df["price_change"] = df["unit_price"] - df["calculated_lag_price"]

df["price_change_pct"] = (
    df["price_change"] / df["calculated_lag_price"]
) * 100

In [25]:
df["price_change_pct"].describe()

count    624.000000
mean      -0.359128
std        7.598358
min      -31.130818
25%       -0.922224
50%        0.000000
75%        0.000000
max       85.714286
Name: price_change_pct, dtype: float64

In [26]:
df["price_direction"] = "No Change"

df.loc[df["price_change"] > 0, "price_direction"] = "Increase"
df.loc[df["price_change"] < 0, "price_direction"] = "Decrease"
df.loc[df["price_change"].isna(), "price_direction"] = "First Observation"

df["price_direction"].value_counts()

price_direction
No Change            327
Decrease             181
Increase             116
First Observation     52
Name: count, dtype: int64

In [27]:
comparable_prices = df[df["price_direction"] != "First Observation"]

(
    comparable_prices["price_direction"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

price_direction
No Change    52.40
Decrease     29.01
Increase     18.59
Name: proportion, dtype: float64

### Pricing Pattern Finding

- Among comparable product-month observations, 52.4% had no price change.
- Price decreases occurred in 29.0% of observations, compared with 18.6% for price increases.
- This suggests pricing was relatively stable, with decreases occurring more often than increases.

In [28]:
df.groupby("price_direction")["qty"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
price_direction,,,
Decrease,181,18.93,12.0
First Observation,52,5.23,3.0
Increase,116,14.01,10.0
No Change,327,13.69,10.0


## 5. Demand Response Analysis

- Compare sales quantity across different price movements.
- Measure within-product quantity changes following price changes.
- Evaluate both percentage and absolute demand changes to avoid misleading conclusions.

### Initial Demand Comparison

- Average and median quantity were highest during price-decrease observations.
- Price-decrease periods had a median quantity of 12 units, compared with 10 units during price increases and unchanged-price periods.
- These are descriptive comparisons and do not establish that price changes caused the differences in demand.

In [29]:
df["previous_qty"] = df.groupby("product_id")["qty"].shift(1)

df["qty_change"] = df["qty"] - df["previous_qty"]

df["qty_change_pct"] = (
    df["qty_change"] / df["previous_qty"]
) * 100

In [30]:
df.groupby("price_direction")["qty_change"].agg(
    ["count", "mean", "median"]
).round(2)

,count,mean,median
price_direction,,,
Decrease,181,3.09,2.0
First Observation,0,NaN,NaN
Increase,116,-6.14,-2.5
No Change,327,1.04,0.0


In [31]:
df.groupby("price_direction")["qty_change_pct"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
price_direction,,,
Decrease,181,102.93,16.67
First Observation,0,NaN,NaN
Increase,116,25.19,-26.14
No Change,327,87.11,0.00


In [32]:
df.nlargest(10, "qty_change_pct")[["product_id","previous_qty","qty","qty_change_pct","price_direction"]]


,product_id,previous_qty,qty,qty_change_pct,price_direction
282,cool3,1.0,76,7500.0,No Change
322,computers1,1.0,31,3000.0,Decrease
152,computers5,2.0,57,2750.0,Decrease
489,watches3,1.0,16,1500.0,No Change
272,watches6,1.0,15,1400.0,Decrease
191,garden6,1.0,14,1300.0,No Change
652,computers2,1.0,12,1100.0,No Change
78,cool4,1.0,12,1100.0,Increase
230,health1,3.0,36,1100.0,No Change
612,watches7,4.0,45,1025.0,Decrease


In [33]:
(df["previous_qty"] <= 5).sum()

np.int64(176)

### Percentage Demand Sensitivity Check

- Percentage quantity changes can be unstable when previous sales volume is small.
- Compare results across several minimum-volume thresholds to check whether conclusions depend on one arbitrary cutoff.

In [34]:
for threshold in [0, 2, 5, 10]:
    temp = df[df["previous_qty"] > threshold]
    result = temp.groupby("price_direction")["qty_change_pct"].agg(
        ["count", "mean", "median"]
    ).round(2)

    print(f"\nPrevious qty > {threshold}")
    print(result)


Previous qty > 0
                 count    mean  median
price_direction                       
Decrease           181  102.93   16.67
Increase           116   25.19  -26.14
No Change          327   87.11    0.00

Previous qty > 2
                 count   mean  median
price_direction                      
Decrease           163  47.43   10.00
Increase           108  -3.97  -30.79
No Change          272  22.00    0.00

Previous qty > 5
                 count   mean  median
price_direction                      
Decrease           139  25.63    0.00
Increase            95 -13.91  -31.58
No Change          214   4.59    0.00

Previous qty > 10
                 count   mean  median
price_direction                      
Decrease            96   1.80  -11.04
Increase            68 -21.09  -37.98
No Change          151  -1.87   -5.56


In [35]:
df["previous_qty"].describe()

count    624.000000
mean      14.967949
std       15.742172
min        1.000000
25%        5.000000
50%       11.000000
75%       19.000000
max      122.000000
Name: previous_qty, dtype: float64

### Validation of Extreme Price Changes

The price-change distribution contains some relatively large movements,
including decreases of approximately 31% and increases of up to 86%.

Before using price-change magnitude in price-response analysis, we inspect
the largest increases and decreases to determine whether they represent
plausible observed price movements or potential data/calculation issues.

In [36]:
df.loc[df["price_change_pct"].notna(),
       ["product_id", "month_year", "calculated_lag_price",
        "unit_price", "price_change_pct", "qty", "previous_qty"]] \
  .sort_values("price_change_pct") \
  .iloc[list(range(5)) + list(range(-5, 0))]

,product_id,month_year,calculated_lag_price,unit_price,price_change_pct,qty,previous_qty
612,watches7,2018-07-01,245.90,169.349318,-31.130818,45,4.0
151,computers5,2017-12-01,167.00,119.900000,-28.203593,2,1.0
636,watches1,2017-06-01,255.61,185.000000,-27.624115,5,3.0
641,watches1,2017-11-01,232.49,174.433333,-24.971683,18,4.0
152,computers5,2018-01-01,119.90,93.094894,-22.356219,57,2.0
36,consoles1,2018-02-01,24.50,32.500000,32.653061,5,16.0
430,garden3,2018-01-01,89.99,123.331667,37.050413,12,14.0
640,watches1,2017-10-01,164.57,232.490000,41.271192,4,20.0
560,garden1,2018-01-01,89.99,127.727273,41.934962,11,27.0
286,cool3,2018-06-01,35.00,65.000000,85.714286,2,4.0


### Price-Response Analysis

- Examine whether the size of a price change is associated with the size and direction of demand change.
- Exclude observations with no price change.
- Treat the results as descriptive associations, not proof of causation.

In [37]:
price_response = df[
    df["price_change_pct"].notna() &
    (df["price_change_pct"] != 0) &
    df["qty_change_pct"].notna()
].copy()

price_response.shape

(297, 40)

### Correlation Between Price Change and Demand Change

- Measure the association between percentage price change and percentage demand change.
- Use this as a descriptive baseline because percentage demand changes can be unstable when previous sales quantities are small.

In [38]:
price_response[["price_change_pct", "qty_change_pct"]].corr().round(3)

,price_change_pct,qty_change_pct
price_change_pct,1.000,-0.214
qty_change_pct,-0.214,1.000


### Absolute Demand Response

- Use absolute quantity change to reduce distortion caused by small previous sales quantities.
- Compare demand changes separately for price increases and price decreases.

In [39]:
price_response.groupby("price_direction")["qty_change"].agg(
    ["count", "mean", "median"]
).round(2)

,count,mean,median
price_direction,,,
Decrease,181,3.09,2.0
Increase,116,-6.14,-2.5


### Demand Response Finding

- Price decreases were associated with a median quantity increase of 2 units.
- Price increases were associated with a median quantity decrease of 2.5 units.
- The results suggest an inverse price-demand relationship, but should not be interpreted as causal.

## 6. Competitor Pricing Analysis

- Compare observed unit prices with the available competitor/reference price fields.
- Measure price gaps and classify whether products are priced above, below, or equal to reference prices.
- Examine whether competitive price position is associated with differences in demand.

In [40]:
df[["unit_price", "comp_1", "ps1", "fp1",
    "comp_2", "ps2", "fp2",
    "comp_3", "ps3", "fp3"]].head(10)

,unit_price,comp_1,ps1,fp1,comp_2,ps2,fp2,comp_3,ps3,fp3
0,45.950000,89.900000,3.9,15.011897,215.000000,4.4,8.760000,45.950000,4.0,15.100000
1,45.950000,89.900000,3.9,14.769216,209.000000,4.4,21.322000,45.950000,4.0,12.933333
2,45.950000,89.900000,3.9,13.993833,205.000000,4.4,22.195932,45.950000,4.0,14.840000
3,45.950000,89.900000,3.9,14.656757,199.509804,4.4,19.412885,45.950000,4.0,14.287500
4,45.950000,89.900000,3.9,18.776522,163.398710,4.4,24.324687,45.950000,4.0,15.100000
5,45.950000,89.900000,3.9,21.572143,45.950000,4.0,15.100000,45.950000,4.0,15.100000
6,40.531818,89.900000,3.9,16.294000,40.531818,4.0,15.832727,40.531818,4.0,15.832727
7,39.990000,88.488235,3.9,18.898333,39.990000,4.0,15.230000,39.990000,4.0,15.230000
8,39.990000,86.900000,3.9,19.383529,39.990000,4.0,16.533684,39.990000,4.0,16.533684
9,39.990000,86.900000,3.9,19.209091,39.990000,4.0,13.749444,39.990000,4.0,13.749444


In [41]:
for col in ["comp_1", "comp_2", "comp_3"]:
    print(col, (df["unit_price"].round(4) == df[col].round(4)).sum())

comp_1 245
comp_2 317
comp_3 287


In [42]:
df.groupby("product_category_name")[["unit_price", "comp_1", "comp_2", "comp_3"]].mean().round(2)

,unit_price,comp_1,comp_2,comp_3
product_category_name,,,,
bed_bath_table,78.63,86.27,100.02,44.30
computers_accessories,119.48,139.46,142.72,102.60
consoles_games,27.03,27.10,26.97,27.03
cool_stuff,107.86,96.48,120.83,86.52
furniture_decor,60.15,75.88,38.73,86.70
garden_tools,80.09,58.06,58.19,59.88
health_beauty,132.31,32.97,90.59,59.73
perfumery,89.35,59.69,119.01,89.35
watches_gifts,164.88,135.53,129.62,172.51


### Competitor Price Gap Analysis

- Calculate the difference between our observed unit price and each competitor/reference price.
- Positive gaps indicate our price is higher; negative gaps indicate our price is lower.

In [43]:
for i in [1, 2, 3]:
    df[f"comp_{i}_gap"] = df["unit_price"] - df[f"comp_{i}"]

df[["comp_1_gap", "comp_2_gap", "comp_3_gap"]].describe().round(2)

,comp_1_gap,comp_2_gap,comp_3_gap
count,676.00,676.00,676.00
mean,27.04,13.57,22.31
std,76.06,68.81,72.74
min,-85.00,-169.05,-163.01
25%,0.00,-0.06,0.00
50%,0.00,0.00,0.00
75%,39.14,13.52,34.91
max,340.01,284.01,305.01


### Competitive Price Position

- Classify each observation as priced above, below, or equal to each competitor/reference price.
- Use these categories to examine competitive positioning and its relationship with demand.

In [44]:
for i in [1, 2, 3]:
    gap = df[f"comp_{i}_gap"]
    df[f"comp_{i}_position"] = "Equal"
    df.loc[gap > 0, f"comp_{i}_position"] = "Above"
    df.loc[gap < 0, f"comp_{i}_position"] = "Below"

for i in [1, 2, 3]:
    print(f"\nCompetitor {i}")
    print(df[f"comp_{i}_position"].value_counts())


Competitor 1
comp_1_position
Above    293
Equal    245
Below    138
Name: count, dtype: int64

Competitor 2
comp_2_position
Equal    317
Above    190
Below    169
Name: count, dtype: int64

Competitor 3
comp_3_position
Equal    287
Above    255
Below    134
Name: count, dtype: int64


### Demand by Competitive Price Position

To explore whether competitive price positioning is associated with demand,
average and median quantities are compared across observations priced above,
below, or equal to each competitor/reference price.

This is a descriptive comparison and does not establish that competitive
price position caused differences in demand.

In [45]:
for i in [1, 2, 3]:
    print(f"\nCompetitor {i}")
    print(df.groupby(f"comp_{i}_position")["qty"].agg(
        ["count", "mean", "median"]
    ).round(2))


Competitor 1
                 count   mean  median
comp_1_position                      
Above              293  13.76    10.0
Below              138  12.46     8.0
Equal              245  16.51    11.0

Competitor 2
                 count   mean  median
comp_2_position                      
Above              190  14.33    10.0
Below              169  18.11    13.0
Equal              317  12.67     9.0

Competitor 3
                 count   mean  median
comp_3_position                      
Above              255  14.39    10.0
Below              134  20.32    13.0
Equal              287  11.87     9.0


In [46]:
category_comp = df.groupby("product_category_name")[
    ["comp_1_gap", "comp_2_gap", "comp_3_gap"]
].median().round(2)

category_comp

,comp_1_gap,comp_2_gap,comp_3_gap
product_category_name,,,
bed_bath_table,0.00,0.00,37.94
computers_accessories,-15.09,0.00,0.00
consoles_games,0.00,0.00,0.00
cool_stuff,0.00,0.00,0.00
furniture_decor,-15.38,0.00,-24.90
garden_tools,0.00,0.00,0.00
health_beauty,45.00,-2.50,21.00
perfumery,0.00,0.00,0.00
watches_gifts,10.20,0.65,-10.00


### Key Findings — Competitor Pricing

- Competitor/reference prices were frequently equal to our observed unit price.
- Our price was more often **above Competitor 1 and Competitor 3** than below them.
- When priced below **Competitor 2**, median quantity sold was **13 units**, compared with **10 units when priced above**.
- When priced below **Competitor 3**, median quantity sold was **13 units**, compared with **10 units when priced above**.
- **Competitor 1 showed no clear demand pattern** based on price position.
- **Health & Beauty** showed notable above-reference pricing:
  - +45.00 vs Competitor 1
  - +21.00 vs Competitor 3
- **Furniture & Decor** showed below-reference pricing:
  - -15.38 vs Competitor 1
  - -24.90 vs Competitor 3
- Competitive positioning varies by category, so a **single pricing strategy may not suit all categories**.
- These relationships show **association, not causation**.

## 7. Product & Category Performance

- Analyze revenue and sales volume across product categories.
- Identify the categories and products contributing most to overall business performance.
- Examine monthly and yearly performance trends.

### Category Performance

- **Goal:** Identify the strongest-performing product categories.
- Compare categories using:
  - **Total Revenue** — monetary contribution.
  - **Total Quantity Sold** — sales volume.
- Revenue and quantity are analyzed together because a category can generate high revenue through either high prices or high sales volume.

In [47]:
category_performance = df.groupby("product_category_name").agg(
    total_revenue=("total_price", "sum"),
    total_qty=("qty", "sum")
).sort_values("total_revenue", ascending=False)

category_performance.round(2)

,total_revenue,total_qty
product_category_name,,
health_beauty,212409.24,1842
watches_gifts,207582.17,1431
garden_tools,163582.54,2398
computers_accessories,142097.95,1167
bed_bath_table,95084.52,1026
cool_stuff,57956.30,558
furniture_decor,56925.16,894
perfumery,20312.52,244
consoles_games,5800.70,239


In [48]:
category_performance["revenue_share_pct"] = (
    category_performance["total_revenue"] /
    category_performance["total_revenue"].sum() * 100
)

category_performance.round(2)

,total_revenue,total_qty,revenue_share_pct
product_category_name,,,
health_beauty,212409.24,1842,22.09
watches_gifts,207582.17,1431,21.58
garden_tools,163582.54,2398,17.01
computers_accessories,142097.95,1167,14.77
bed_bath_table,95084.52,1026,9.89
cool_stuff,57956.30,558,6.03
furniture_decor,56925.16,894,5.92
perfumery,20312.52,244,2.11
consoles_games,5800.70,239,0.60


### Category Performance findings

- **Health & Beauty** generated the highest revenue (**212,409.24**) and contributed **22.09%** of total revenue.
- **Watches & Gifts** ranked second with **207,582.17 revenue (21.58%)**.
- **Garden Tools** had the highest sales volume (**2,398 units**) and contributed **17.01%** of revenue.
- The **top 3 categories contributed 60.68%** of total revenue.
- The **top 4 categories contributed 75.45%**, showing that revenue is concentrated in a few categories.
- **Consoles & Games** was the weakest category, contributing only **0.60%** of total revenue.
- High sales volume does not necessarily mean high revenue because **unit prices differ across categories**.

### Sales Performance Over Time

- Compare monthly and yearly revenue and sales volume.
- Identify the strongest and weakest observed periods.
- Review time-based patterns without assuming seasonality from the limited time range.

In [49]:
monthly_performance = df.groupby("month_year").agg(
    total_revenue=("total_price", "sum"),
    total_qty=("qty", "sum")
).sort_index()

monthly_performance.round(2)

,total_revenue,total_qty
month_year,,
2017-01-01,2864.19,9
2017-02-01,3584.11,35
2017-03-01,10204.38,101
2017-04-01,11524.62,121
2017-05-01,21843.33,222
2017-06-01,23245.24,233
2017-07-01,41049.89,403
2017-08-01,49550.41,495
2017-09-01,44826.60,450


In [50]:
yearly_performance = df.groupby("year").agg(
    total_revenue=("total_price", "sum"),
    total_qty=("qty", "sum")
)

yearly_performance.round(2)

,total_revenue,total_qty
year,,
2017,409583.28,4348
2018,552167.82,5451


### Time-Based Performance Findings

- November 2017 recorded the highest sales volume with 1,051 units and 87,009.77 in revenue.
- May 2018 recorded the highest monthly revenue at 91,223.46.
- Performance remained relatively strong from January to May 2018 before declining through August 2018.
- Observed 2018 revenue and quantity exceeded 2017 despite containing only January–August data.
- Yearly comparisons have unequal coverage, so these results should not be interpreted as full-year growth or reliable seasonality.

### Product Performance

- Compare individual products by total revenue and sales volume.
- Identify top and low-performing products and differences between revenue and volume performance.

In [51]:
product_performance = df.groupby("product_id").agg(
    category=("product_category_name", "first"),
    total_revenue=("total_price", "sum"),
    total_qty=("qty", "sum")
).sort_values("total_revenue", ascending=False)

product_performance.head(10).round(2)

,category,total_revenue,total_qty
product_id,,,
health2,health_beauty,63885.00,195
health5,health_beauty,54730.20,156
computers4,computers_accessories,46916.51,341
bed2,bed_bath_table,42938.66,487
computers6,computers_accessories,41082.60,274
watches6,watches_gifts,37683.42,323
furniture2,furniture_decor,37608.90,527
watches1,watches_gifts,31786.82,194
watches7,watches_gifts,31623.81,123


In [52]:
product_performance.sort_values(
    "total_qty", ascending=False
).head(10).round(2)

,category,total_revenue,total_qty
product_id,,,
furniture2,furniture_decor,37608.90,527
bed2,bed_bath_table,42938.66,487
garden6,garden_tools,26577.22,484
garden10,garden_tools,21390.69,391
garden4,garden_tools,21056.80,388
garden9,garden_tools,20387.20,373
computers4,computers_accessories,46916.51,341
watches6,watches_gifts,37683.42,323
health9,health_beauty,6325.19,281


In [53]:
product_performance.sort_values(
    "total_revenue"
).head(10).round(2)

,category,total_revenue,total_qty
product_id,,,
consoles2,consoles_games,2384.00,97
consoles1,consoles_games,3416.70,142
furniture3,furniture_decor,3507.95,97
cool3,cool_stuff,3656.50,114
health4,health_beauty,4096.30,137
bed1,bed_bath_table,5831.77,143
bed4,bed_bath_table,6011.77,131
furniture1,furniture_decor,6162.11,169
garden7,garden_tools,6220.90,115


### Product Performance Findings

- `health2` generated the highest product revenue at 63,885.00, followed by `health5` at 54,730.20.
- `furniture2` recorded the highest sales volume with 527 units, while `bed2` ranked second with 487 units.
- Garden Tools had several products among the highest-volume products, supporting its strong category-level sales volume.
- High-volume products were not always the highest-revenue products, showing the importance of evaluating both revenue and quantity.
- `consoles2` and `consoles1` generated the lowest product revenues, consistent with the weak overall performance of Consoles & Games.

### Customer & Calendar Context

- Review customer activity and the available monthly calendar variables.
- Examine whether customer activity is associated with sales quantity and revenue.
- Interpret calendar fields carefully because they represent monthly counts rather than binary indicators.

In [54]:
df[["customers", "weekday", "weekend", "holiday"]].describe().round(2)

,customers,weekday,weekend,holiday
count,676.00,676.00,676.00,676.00
mean,81.03,21.77,8.66,1.49
std,62.06,0.99,0.71,0.94
min,1.00,20.00,8.00,0.00
25%,34.00,21.00,8.00,1.00
50%,62.00,22.00,9.00,1.00
75%,116.00,23.00,9.00,2.00
max,339.00,23.00,10.00,4.00


In [55]:
df[["customers", "qty", "total_price"]].corr().round(3)

,customers,qty,total_price
customers,1.000,0.442,0.386
qty,0.442,1.000,0.750
total_price,0.386,0.750,1.000


### Customer Activity Findings

- Customer activity had a moderate positive correlation with quantity sold (`r = 0.442`).
- Customer activity also had a positive correlation with revenue (`r = 0.386`).
- Higher customer activity therefore tended to coincide with stronger sales performance.
- These relationships represent association rather than causation.
- Weekday, weekend, and holiday variables represent monthly calendar counts and were not treated as simple demand indicators.

## 8. Statistical Validation

- Test whether demand changes differ significantly between price-increase and price-decrease observations.
- Use a simple non-parametric test because quantity-change distributions may contain outliers and may not be normally distributed.

In [56]:
from scipy.stats import mannwhitneyu

increase_demand = price_response.loc[
    price_response["price_direction"] == "Increase", "qty_change"
].dropna()

decrease_demand = price_response.loc[
    price_response["price_direction"] == "Decrease", "qty_change"
].dropna()

stat, p_value = mannwhitneyu(
    decrease_demand,
    increase_demand,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", round(stat, 2))
print("p-value:", f"{p_value:.6g}")

Mann-Whitney U statistic: 13442.5
p-value: 4.50776e-05


### Statistical Test Finding

- The Mann–Whitney U test produced a statistic of 13,442.5 and a p-value of 0.000045.
- Since the p-value is below 0.05, quantity changes differed significantly between price-increase and price-decrease observations.
- Combined with the earlier demand analysis, price decreases were associated with higher quantity changes, while price increases were associated with lower quantity changes.
- This provides statistical evidence of an association between price movement and demand response, but does not establish causation.

## 9. Pricing Opportunity Identification

- Combine product performance with competitor/reference price gaps to identify products that may deserve pricing review.
- Focus on business contribution and competitive positioning rather than automatically recommending price changes.

In [57]:
pricing_opportunities = df.groupby("product_id").agg(
    category=("product_category_name", "first"),
    avg_price=("unit_price", "mean"),
    total_revenue=("total_price", "sum"),
    total_qty=("qty", "sum"),
    avg_comp_1_gap=("comp_1_gap", "mean"),
    avg_comp_2_gap=("comp_2_gap", "mean"),
    avg_comp_3_gap=("comp_3_gap", "mean")
).reset_index()

pricing_opportunities["avg_comp_gap"] = pricing_opportunities[
    ["avg_comp_1_gap", "avg_comp_2_gap", "avg_comp_3_gap"]
].mean(axis=1)

pricing_opportunities.sort_values(
    ["total_revenue", "avg_comp_gap"],
    ascending=[False, False]
).head(15).round(2)

,product_id,category,avg_price,total_revenue,total_qty,avg_comp_1_gap,avg_comp_2_gap,avg_comp_3_gap,avg_comp_gap
34,health2,health_beauty,326.99,63885.00,195,279.51,241.11,268.00,262.87
37,health5,health_beauty,350.74,54730.20,156,295.39,224.44,290.25,270.03
8,computers4,computers_accessories,141.58,46916.51,341,0.00,-9.09,25.76,5.56
1,bed2,bed_bath_table,86.77,42938.66,487,0.00,-28.55,37.03,2.83
10,computers6,computers_accessories,149.96,41082.60,274,20.45,0.00,68.39,29.61
49,watches6,watches_gifts,132.53,37683.42,323,0.00,9.42,-37.12,-9.23
19,furniture2,furniture_decor,77.51,37608.90,527,0.00,37.95,-13.76,8.06
44,watches1,watches_gifts,180.63,31786.82,194,30.57,32.47,0.00,21.01
50,watches7,watches_gifts,305.69,31623.81,123,159.13,140.54,135.18,144.95
48,watches5,watches_gifts,213.90,30467.50,143,73.67,72.62,38.61,61.63


### Pricing Opportunity Findings

- `health2` and `health5`, the two highest-revenue products, also showed very large positive average competitor/reference price gaps.
- `watches7`, `bed5`, and `watches5` also combined meaningful revenue with substantial above-reference pricing.
- Some strong-selling products, including `watches6` and `watches4`, were priced below the average reference level.
- High-revenue products with large positive price gaps may warrant competitive pricing review, while strong products priced below references may warrant margin-opportunity investigation.
- These products are candidates for further pricing analysis rather than automatic price-change recommendations.

## 10. Proposed Price Experiment

- Select a high-volume product with a meaningful pricing opportunity for a controlled price test.
- Keep the current price for a comparable control group and apply a small predefined price change to a treatment group.
- Track unit sales, revenue, and margin during the test period.
- Control for major factors such as promotions, inventory availability, and seasonal effects where possible.
- Compare treatment and control performance to determine whether the price change improves business outcomes.
- Validate statistically significant results before considering broader implementation.

**Note:** The current dataset is observational, so no A/B test was performed. This experiment is proposed as a method for validating future pricing decisions.

## 11. Analysis Limitations

- The dataset is observational, so price-demand relationships should not be interpreted as causal.
- Some product histories contain missing months, so previous observations are not always consecutive calendar months.
- Competitor/reference price fields frequently match the observed unit price, so their exact business interpretation should be treated cautiously.
- Percentage demand changes can be unstable when previous sales quantities are small.
- Factors such as promotions, inventory availability, marketing activity, and product-specific seasonality may not be fully represented.
- The 2017 and 2018 periods have unequal calendar coverage, limiting direct full-year comparisons.
- Identified pricing opportunities require further business validation before implementation.

## 12. Project Summary

RetailPulse analyzed product-level retail pricing, demand, competitor positioning, and business performance using Python.

Key analytical steps included:

- Data cleaning and validation
- Price-change and demand-response analysis
- Competitor/reference price benchmarking
- Category, product, and time-based performance analysis
- Statistical validation of demand differences
- Identification of potential pricing-review opportunities
- Design of a proposed controlled price experiment

The analysis identified meaningful pricing and demand patterns while distinguishing statistical association from causation.

In [58]:
# Create a MySQL-friendly version of the cleaned dataset
mysql_df = df.copy()

mysql_df["month_year"] = pd.to_datetime(
    mysql_df["month_year"]
).dt.strftime("%Y-%m-%d")

mysql_df = mysql_df.replace([np.inf, -np.inf], np.nan)

mysql_df.to_csv(
    "../data/retailpulse_mysql.csv",
    index=False,
    na_rep=""
)

print(mysql_df.shape)

(676, 46)
